<a href="https://colab.research.google.com/github/salochaud/DatosMasivos/blob/main/Tarea_4_5_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyspark -q

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

spark = SparkSession.builder \
    .appName("Tarea 4 y 5 ") \
    .getOrCreate()

In [3]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/content/players_22.csv")

df.show(5)

+---------+--------------------+-----------------+--------------------+----------------+-------+---------+---------+--------+---+----------+---------+---------+------------+-------------------+--------------------+------------+-------------+------------------+----------------+-----------+-------------------------+--------------+----------------+--------------+---------------+--------------------+--------------+---------+-----------+------------------------+-----------+---------+---------+------------------+--------------------+--------------------+----+--------+-------+---------+---------+------+------------------+-------------------+--------------------------+-----------------------+-----------------+---------------+-----------+-----------------+------------------+------------------+---------------------+---------------------+----------------+------------------+----------------+----------------+-------------+-------------+--------------+----------------+--------------------+----------

In [4]:
df = df.filter(
    col("overall").isNotNull() &
    col("pace").isNotNull() &
    col("shooting").isNotNull() &
    col("passing").isNotNull() &
    col("dribbling").isNotNull() &
    col("defending").isNotNull() &
    col("physic").isNotNull() &
    col("potential").isNotNull() &
    col("age").isNotNull()
)

df.show(5)

+---------+--------------------+-----------------+--------------------+----------------+-------+---------+---------+--------+---+----------+---------+---------+------------+-------------------+--------------------+------------+-------------+------------------+----------------+-----------+-------------------------+--------------+----------------+--------------+---------------+--------------------+--------------+---------+-----------+------------------------+-----------+---------+---------+------------------+--------------------+--------------------+----+--------+-------+---------+---------+------+------------------+-------------------+--------------------------+-----------------------+-----------------+---------------+-----------+-----------------+------------------+------------------+---------------------+---------------------+----------------+------------------+----------------+----------------+-------------+-------------+--------------+----------------+--------------------+----------

In [5]:
assembler = VectorAssembler(
    inputCols=["overall", "potential", "age", "pace", "shooting", "passing", "dribbling", "defending", "physic"],
    outputCol="features"
)

df_features = assembler.transform(df)
df_features.select("short_name", "overall", "features").show(5)

+-----------------+-------+--------------------+
|       short_name|overall|            features|
+-----------------+-------+--------------------+
|         L. Messi|     93|[93.0,93.0,34.0,8...|
|   R. Lewandowski|     92|[92.0,92.0,32.0,7...|
|Cristiano Ronaldo|     91|[91.0,91.0,36.0,8...|
|        Neymar Jr|     91|[91.0,91.0,29.0,9...|
|     K. De Bruyne|     91|[91.0,91.0,30.0,7...|
+-----------------+-------+--------------------+
only showing top 5 rows


In [6]:
scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withStd=True, withMean=True)
scaler_model = scaler.fit(df_features)
df_scaled = scaler_model.transform(df_features)

df_scaled.select("short_name", "scaled_features").show(5)

+-----------------+--------------------+
|       short_name|     scaled_features|
+-----------------+--------------------+
|         L. Messi|[3.99632614079360...|
|   R. Lewandowski|[3.84863828547498...|
|Cristiano Ronaldo|[3.70095043015635...|
|        Neymar Jr|[3.70095043015635...|
|     K. De Bruyne|[3.70095043015635...|
+-----------------+--------------------+
only showing top 5 rows


In [7]:
kmeans = KMeans(featuresCol="scaled_features", predictionCol="cluster", k=3, seed=42)
kmeans_model = kmeans.fit(df_scaled)

In [8]:
df_cluster = kmeans_model.transform(df_scaled)
df_cluster.select("short_name", "nationality_name", "club_name", "overall", "potential", "age", "cluster").show(10)

+-----------------+----------------+-------------------+-------+---------+---+-------+
|       short_name|nationality_name|          club_name|overall|potential|age|cluster|
+-----------------+----------------+-------------------+-------+---------+---+-------+
|         L. Messi|       Argentina|Paris Saint-Germain|     93|       93| 34|      1|
|   R. Lewandowski|          Poland|  FC Bayern München|     92|       92| 32|      1|
|Cristiano Ronaldo|        Portugal|  Manchester United|     91|       91| 36|      1|
|        Neymar Jr|          Brazil|Paris Saint-Germain|     91|       91| 29|      1|
|     K. De Bruyne|         Belgium|    Manchester City|     91|       91| 30|      1|
|        K. Mbappé|          France|Paris Saint-Germain|     91|       95| 22|      1|
|          H. Kane|         England|  Tottenham Hotspur|     90|       90| 27|      1|
|         N. Kanté|          France|            Chelsea|     90|       90| 30|      1|
|       K. Benzema|          France|     Re

In [9]:
evaluator = ClusteringEvaluator(featuresCol="scaled_features", predictionCol="cluster", metricName="silhouette")
silhouette = evaluator.evaluate(df_cluster)
print(f"Silhouette Score: {silhouette}")

Silhouette Score: 0.3486672635024078


In [10]:
df_cluster.groupBy("cluster").agg({
    "overall": "avg",
    "potential": "avg",
    "age": "avg",
    "pace": "avg",
    "shooting": "avg",
    "defending": "avg"
}).orderBy("cluster").show()

+-------+------------------+-----------------+-----------------+-----------------+-----------------+------------------+
|cluster|     avg(shooting)|   avg(defending)|     avg(overall)|   avg(potential)|        avg(pace)|          avg(age)|
+-------+------------------+-----------------+-----------------+-----------------+-----------------+------------------+
|      0| 54.66536675951718|37.16508820798514| 61.1949860724234|69.98700092850511| 71.5920148560817|22.188857938718662|
|      1| 61.95128166691651|56.06086044071353|71.70694048868236|74.34072852645781|71.49527806925498|27.125018737820415|
|      2|37.184913878439914|61.44882201544249|63.38467630172243|68.48267669768363|60.27578697287666| 25.48742823203326|
+-------+------------------+-----------------+-----------------+-----------------+-----------------+------------------+

